In [2]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
import pandas as pd
import random

np.random.seed(42)
random.seed(42)

def build_reg_model(params):
    return RandomForestRegressor(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        min_samples_split=params['min_samples_split'],
        min_samples_leaf=params['min_samples_leaf'],
        max_features=params['max_features'],
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )

def mean_relative_error(y_true, y_pred):
    mask = y_true != 0
    if np.sum(mask) == 0:
        return 0.0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mre = mean_relative_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'mae': mae, 'mse': mse, 'rmse': rmse, 'mre': mre, 'r2': r2}

print("=" * 60)
print("Step 1/4: Data Preprocessing (IMS Prediction)")
print("-" * 60)

pattern = pd.read_csv('4-pattern2.csv', header=None, encoding='utf-8-sig').values.astype('float64')
pattern = np.where(np.isinf(pattern), np.nan, pattern)
mean_val = np.nanmean(pattern) if not np.isnan(np.nanmean(pattern)) else 0
pattern = np.nan_to_num(pattern, nan=mean_val)

min_vals = np.min(pattern, axis=0)
max_vals = np.max(pattern, axis=0)
range_vals = np.where(max_vals - min_vals == 0, 1, max_vals - min_vals)
pattern_normalized = (pattern - min_vals) / range_vals

scaler_spectral = StandardScaler()
pattern_scaled = scaler_spectral.fit_transform(pattern_normalized)

print("\nLoading image features...")
image_features = pd.read_csv('image_features_mobilenet_large.csv', header=None).values.astype('float64')
scaler_image = StandardScaler()
image_features_scaled = scaler_image.fit_transform(image_features)

label_data = pd.read_csv('label_rIMS2-2.csv', header=None, encoding='utf-8-sig').values.astype('float64')
groups = label_data[:, 0]
label_c_original = label_data[:, 1]
label_c = np.exp(label_c_original)
total_samples = len(label_c)

unique_groups = np.unique(groups)
test_size_groups = int(0.2 * len(unique_groups))
train_groups, test_groups = train_test_split(
    unique_groups,
    test_size=test_size_groups,
    random_state=42
)

train_mask = np.isin(groups, train_groups)
test_mask = np.isin(groups, test_groups)
train_indices = np.where(train_mask)[0]
test_indices = np.where(test_mask)[0]

X_train_raw_spectral = pattern_scaled[train_mask]
X_test_raw_spectral = pattern_scaled[test_mask]
y_train = label_c[train_mask]
y_test = label_c[test_mask]

pca_spectral = PCA(n_components=6)
X_train_spectral_pca = pca_spectral.fit_transform(X_train_raw_spectral)
X_test_spectral_pca = pca_spectral.transform(X_test_raw_spectral)
print(f"Spectral PCA completed: 6 components retained (fixed)")

pca_image = PCA(n_components=0.95)
pattern_image_pca_full = pca_image.fit_transform(image_features_scaled)
image_pca_variance = pca_image.explained_variance_ratio_
print(f"Image PCA completed: {pca_image.n_components_} components retained, cumulative variance ratio: {np.sum(image_pca_variance):.4f}")
print(f"Top 5 image PC variance ratios: {[round(v, 4) for v in image_pca_variance[:5]]}")

X_train_image_pca_full = pattern_image_pca_full[train_mask]
X_test_image_pca_full = pattern_image_pca_full[test_mask]

GAP_THRESHOLD = 0.15

print("\n" + "=" * 60)
print("Step 2/4: Pure Spectral Baseline Model (IMS, 6 PCA)")
print("-" * 60)

ALL_PREDICTIONS = []

baseline_best_params = {
    'max_depth': None,
    'max_features': 'sqrt',
    'min_samples_leaf': 1,
    'min_samples_split': 2,
    'n_estimators': 200
}

X_tr_base, X_val_base, y_tr_base, y_val_base, tr_idx_base, val_idx_base = train_test_split(
    X_train_spectral_pca, y_train, train_indices,
    test_size=0.2,
    random_state=42
)
baseline_model = build_reg_model(baseline_best_params)
baseline_model.fit(X_tr_base, y_tr_base)

y_tr_pred_base = baseline_model.predict(X_tr_base)
y_val_pred_base = baseline_model.predict(X_val_base)
y_train_pred_base = baseline_model.predict(X_train_spectral_pca)
y_test_pred_base = baseline_model.predict(X_test_spectral_pca)

tr_metrics_base = calculate_metrics(y_tr_base, y_tr_pred_base)
val_metrics_base = calculate_metrics(y_val_base, y_val_pred_base)
train_metrics_base = calculate_metrics(y_train, y_train_pred_base)
test_metrics_base = calculate_metrics(y_test, y_test_pred_base)

train_val_gap_base = abs(train_metrics_base['r2'] - val_metrics_base['r2'])
train_test_gap_base = abs(train_metrics_base['r2'] - test_metrics_base['r2'])
is_valid_base = (train_val_gap_base < GAP_THRESHOLD) and (train_test_gap_base < GAP_THRESHOLD)

BASELINE_RESULT = {
    'Train_R2': round(train_metrics_base['r2'], 6),
    'Train_MAE': round(train_metrics_base['mae'], 6),
    'Train_MSE': round(train_metrics_base['mse'], 6),
    'Train_MRE': round(train_metrics_base['mre'], 6),
    'Train_RMSE': round(train_metrics_base['rmse'], 6),
    'Val_R2': round(val_metrics_base['r2'], 6),
    'Val_MAE': round(val_metrics_base['mae'], 6),
    'Val_MSE': round(val_metrics_base['mse'], 6),
    'Val_MRE': round(val_metrics_base['mre'], 6),
    'Val_RMSE': round(val_metrics_base['rmse'], 6),
    'Test_R2': round(test_metrics_base['r2'], 6),
    'Test_MAE': round(test_metrics_base['mae'], 6),
    'Test_MSE': round(test_metrics_base['mse'], 6),
    'Test_MRE': round(test_metrics_base['mre'], 6),
    'Test_RMSE': round(test_metrics_base['rmse'], 6),
    'Train_Val_Gap': round(train_val_gap_base, 6),
    'Train_Test_Gap': round(train_test_gap_base, 6),
    'Best_Params': str(baseline_best_params),
    'Is_Valid': is_valid_base
}

print("Pure Spectral Baseline (IMS, 6 PCA) Metrics:")
print(f"[Train] R²: {BASELINE_RESULT['Train_R2']:.6f} | MAE: {BASELINE_RESULT['Train_MAE']:.6f} | MSE: {BASELINE_RESULT['Train_MSE']:.6f} | MRE: {BASELINE_RESULT['Train_MRE']:.6f} | RMSE: {BASELINE_RESULT['Train_RMSE']:.6f}")
print(f"[Val] R²: {BASELINE_RESULT['Val_R2']:.6f} | MAE: {BASELINE_RESULT['Val_MAE']:.6f} | MSE: {BASELINE_RESULT['Val_MSE']:.6f} | MRE: {BASELINE_RESULT['Val_MRE']:.6f} | RMSE: {BASELINE_RESULT['Val_RMSE']:.6f}")
print(f"[Test] R²: {BASELINE_RESULT['Test_R2']:.6f} | MAE: {BASELINE_RESULT['Test_MAE']:.6f} | MSE: {BASELINE_RESULT['Test_MSE']:.6f} | MRE: {BASELINE_RESULT['Test_MRE']:.6f} | RMSE: {BASELINE_RESULT['Test_RMSE']:.6f}")
print(f"Train-Val Gap: {BASELINE_RESULT['Train_Val_Gap']:.6f} | Train-Test Gap: {BASELINE_RESULT['Train_Test_Gap']:.6f} | Valid: {BASELINE_RESULT['Is_Valid']}")

all_pred_base = np.full(total_samples, np.nan)
data_type_base = np.full(total_samples, "Unused")
all_pred_base[tr_idx_base] = y_tr_pred_base
all_pred_base[val_idx_base] = y_val_pred_base
all_pred_base[test_indices] = y_test_pred_base
data_type_base[tr_idx_base] = "Training Set"
data_type_base[val_idx_base] = "Validation Set"
data_type_base[test_indices] = "Test Set"
for idx in range(total_samples):
    ALL_PREDICTIONS.append({
        "Original_Index": idx + 1,
        "True_Label": round(label_c_original[idx], 6),
        "Model_Type": "Spectral_Baseline",
        "Predicted_Label": round(np.log(all_pred_base[idx]), 6) if not np.isnan(all_pred_base[idx]) else np.nan,
        "Data_Set_Type": data_type_base[idx]
    })

print("\n" + "=" * 60)
print("Step 3/4: Fused Model with Best Parameters (K=2)")
print("-" * 60)

best_k = 2
best_selected_indices = [6, 8]
best_params = {'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 300}

selected_variance = image_pca_variance[best_selected_indices]
cumulative_variance = np.sum(selected_variance)
print(f"Selected image PC indices: {best_selected_indices}")
print(f"Selected PC variance ratios: {[round(v, 4) for v in selected_variance]}")
print(f"Cumulative variance ratio: {cumulative_variance:.4f}")

selector = SelectKBest(score_func=f_regression, k=best_k)
X_train_image_selected = selector.fit_transform(X_train_image_pca_full, y_train)
X_test_image_selected = selector.transform(X_test_image_pca_full)

X_train_fused = np.hstack([X_train_spectral_pca, X_train_image_selected])
X_test_fused = np.hstack([X_test_spectral_pca, X_test_image_selected])

X_tr, X_val, y_tr, y_val, tr_idx, val_idx = train_test_split(
    X_train_fused, y_train, train_indices,
    test_size=0.2,
    random_state=42
)

final_model = build_reg_model(best_params)
final_model.fit(X_tr, y_tr)

y_tr_pred = final_model.predict(X_tr)
y_val_pred = final_model.predict(X_val)
y_train_pred = final_model.predict(X_train_fused)
y_test_pred = final_model.predict(X_test_fused)

tr_metrics = calculate_metrics(y_tr, y_tr_pred)
val_metrics = calculate_metrics(y_val, y_val_pred)
train_metrics = calculate_metrics(y_train, y_train_pred)
test_metrics = calculate_metrics(y_test, y_test_pred)

train_val_gap = abs(train_metrics['r2'] - val_metrics['r2'])
train_test_gap = abs(train_metrics['r2'] - test_metrics['r2'])
is_valid = (train_val_gap < GAP_THRESHOLD) and (train_test_gap < GAP_THRESHOLD)

BEST_OVERALL = {
    'K': best_k,
    'Selected_Image_PCs': str(best_selected_indices),
    'Cumulative_Variance_Ratio': round(cumulative_variance, 4),
    'Best_Params': str(best_params),
    'CV_R2': 'N/A (direct best params)',
    'Train_R2': round(train_metrics['r2'], 6),
    'Train_MAE': round(train_metrics['mae'], 6),
    'Train_MSE': round(train_metrics['mse'], 6),
    'Train_MRE': round(train_metrics['mre'], 6),
    'Train_RMSE': round(train_metrics['rmse'], 6),
    'Val_R2': round(val_metrics['r2'], 6),
    'Val_MAE': round(val_metrics['mae'], 6),
    'Val_MSE': round(val_metrics['mse'], 6),
    'Val_MRE': round(val_metrics['mre'], 6),
    'Val_RMSE': round(val_metrics['rmse'], 6),
    'Test_R2': round(test_metrics['r2'], 6),
    'Test_MAE': round(test_metrics['mae'], 6),
    'Test_MSE': round(test_metrics['mse'], 6),
    'Test_MRE': round(test_metrics['mre'], 6),
    'Test_RMSE': round(test_metrics['rmse'], 6),
    'Train_Val_Gap': round(train_val_gap, 6),
    'Train_Test_Gap': round(train_test_gap, 6),
    'Is_Valid': is_valid
}

all_pred = np.full(total_samples, np.nan)
data_type = np.full(total_samples, "Unused")
all_pred[tr_idx] = y_tr_pred
all_pred[val_idx] = y_val_pred
all_pred[test_indices] = y_test_pred
data_type[tr_idx] = "Training Set"
data_type[val_idx] = "Validation Set"
data_type[test_indices] = "Test Set"
for idx in range(total_samples):
    ALL_PREDICTIONS.append({
        "Original_Index": idx + 1,
        "True_Label": round(label_c_original[idx], 6),
        "Model_Type": "Fused_K=2",
        "Predicted_Label": round(np.log(all_pred[idx]), 6) if not np.isnan(all_pred[idx]) else np.nan,
        "Data_Set_Type": data_type[idx]
    })

print(f"Test R²: {test_metrics['r2']:.6f} | Train-Test Gap: {train_test_gap:.6f} | Valid: {is_valid}")

print("\n" + "=" * 60)
print("Step 4/4: Final Results Summary (IMS)")
print("-" * 60)

results_list = [BEST_OVERALL]
results_df = pd.DataFrame(results_list)
predictions_df = pd.DataFrame(ALL_PREDICTIONS)
predictions_df = predictions_df.sort_values(by=['Original_Index', 'Model_Type']).reset_index(drop=True)

summary_cols = ['K', 'Cumulative_Variance_Ratio', 'CV_R2', 'Test_R2', 'Test_MAE', 'Test_MSE', 'Test_RMSE', 'Train_Test_Gap', 'Is_Valid']
print(results_df[summary_cols].to_string(index=False))

print("\n" + "=" * 60)
print("Best Overall Fused Model Results (IMS)")
print("-" * 60)
print(f"Optimal K value: {BEST_OVERALL['K']}")
print(f"Selected image PC indices: {BEST_OVERALL['Selected_Image_PCs']}")
print(f"Cumulative variance ratio: {BEST_OVERALL['Cumulative_Variance_Ratio']:.4f}")
print(f"Optimal parameters: {BEST_OVERALL['Best_Params']}")
print("\nDetailed Performance Metrics:")
print(f"[Train] R²: {BEST_OVERALL['Train_R2']:.6f} | MAE: {BEST_OVERALL['Train_MAE']:.6f} | MSE: {BEST_OVERALL['Train_MSE']:.6f} | MRE: {BEST_OVERALL['Train_MRE']:.6f} | RMSE: {BEST_OVERALL['Train_RMSE']:.6f}")
print(f"[Val] R²: {BEST_OVERALL['Val_R2']:.6f} | MAE: {BEST_OVERALL['Val_MAE']:.6f} | MSE: {BEST_OVERALL['Val_MSE']:.6f} | MRE: {BEST_OVERALL['Val_MRE']:.6f} | RMSE: {BEST_OVERALL['Val_RMSE']:.6f}")
print(f"[Test] R²: {BEST_OVERALL['Test_R2']:.6f} | MAE: {BEST_OVERALL['Test_MAE']:.6f} | MSE: {BEST_OVERALL['Test_MSE']:.6f} | MRE: {BEST_OVERALL['Test_MRE']:.6f} | RMSE: {BEST_OVERALL['Test_RMSE']:.6f}")
print(f"Train-Val Gap: {BEST_OVERALL['Train_Val_Gap']:.6f} | Train-Test Gap: {BEST_OVERALL['Train_Test_Gap']:.6f} | Valid: {BEST_OVERALL['Is_Valid']}")

train_improve_r2 = (BEST_OVERALL['Train_R2'] - BASELINE_RESULT['Train_R2']) * 100
train_improve_mae = (BASELINE_RESULT['Train_MAE'] - BEST_OVERALL['Train_MAE']) * 100
train_improve_mse = (BASELINE_RESULT['Train_MSE'] - BEST_OVERALL['Train_MSE']) * 100
train_improve_mre = (BASELINE_RESULT['Train_MRE'] - BEST_OVERALL['Train_MRE']) * 100
train_improve_rmse = (BASELINE_RESULT['Train_RMSE'] - BEST_OVERALL['Train_RMSE']) * 100

val_improve_r2 = (BEST_OVERALL['Val_R2'] - BASELINE_RESULT['Val_R2']) * 100
val_improve_mae = (BASELINE_RESULT['Val_MAE'] - BEST_OVERALL['Val_MAE']) * 100
val_improve_mse = (BASELINE_RESULT['Val_MSE'] - BEST_OVERALL['Val_MSE']) * 100
val_improve_mre = (BASELINE_RESULT['Val_MRE'] - BEST_OVERALL['Val_MRE']) * 100
val_improve_rmse = (BASELINE_RESULT['Val_RMSE'] - BEST_OVERALL['Val_RMSE']) * 100

test_improve_r2 = (BEST_OVERALL['Test_R2'] - BASELINE_RESULT['Test_R2']) * 100
test_improve_mae = (BASELINE_RESULT['Test_MAE'] - BEST_OVERALL['Test_MAE']) * 100
test_improve_mse = (BASELINE_RESULT['Test_MSE'] - BEST_OVERALL['Test_MSE']) * 100
test_improve_mre = (BASELINE_RESULT['Test_MRE'] - BEST_OVERALL['Test_MRE']) * 100
test_improve_rmse = (BASELINE_RESULT['Test_RMSE'] - BEST_OVERALL['Test_RMSE']) * 100

print("\nComparison with Pure Spectral Baseline (All Datasets):")
print("-"*50)
print(f"Training Set")
print(f"Baseline Train R²: {BASELINE_RESULT['Train_R2']:.6f} | Best Fused Train R²: {BEST_OVERALL['Train_R2']:.6f}")
print(f"Improvement: R² {train_improve_r2:+.2f}% | MAE {train_improve_mae:+.2f}% | MSE {train_improve_mse:+.2f}% | MRE {train_improve_mre:+.2f}% | RMSE {train_improve_rmse:+.2f}%")

print(f"\nValidation Set")
print(f"Baseline Val R²: {BASELINE_RESULT['Val_R2']:.6f} | Best Fused Val R²: {BEST_OVERALL['Val_R2']:.6f}")
print(f"Improvement: R² {val_improve_r2:+.2f}% | MAE {val_improve_mae:+.2f}% | MSE {val_improve_mse:+.2f}% | MRE {val_improve_mre:+.2f}% | RMSE {val_improve_rmse:+.2f}%")

print(f"\nTest Set")
print(f"Baseline Test R²: {BASELINE_RESULT['Test_R2']:.6f} | Best Fused Test R²: {BEST_OVERALL['Test_R2']:.6f}")
print(f"Improvement: R² {test_improve_r2:+.2f}% | MAE {test_improve_mae:+.2f}% | MSE {test_improve_mse:+.2f}% | MRE {test_improve_mre:+.2f}% | RMSE {test_improve_rmse:+.2f}%")

with pd.ExcelWriter('Image_Feature_K_Selection_Results_IMS_Tuned.xlsx', engine='openpyxl') as writer:
    pd.DataFrame([BASELINE_RESULT]).to_excel(writer, sheet_name='Spectral_Baseline', index=False)
    results_df.to_excel(writer, sheet_name='All_K_Results', index=False)
    pd.DataFrame([BEST_OVERALL]).to_excel(writer, sheet_name='Best_Fused_Result', index=False)
    predictions_df.to_excel(writer, sheet_name='All_Sample_Predictions', index=False)

print(f"\nAll results saved to: Image_Feature_K_Selection_Results_IMS_Tuned.xlsx")
print("=" * 60)

Step 1/4: Data Preprocessing (IMS Prediction)
------------------------------------------------------------

Loading image features...
Spectral PCA completed: 6 components retained (fixed)
Image PCA completed: 186 components retained, cumulative variance ratio: 0.9503
Top 5 image PC variance ratios: [np.float64(0.1658), np.float64(0.0762), np.float64(0.0614), np.float64(0.0468), np.float64(0.0415)]

Step 2/4: Pure Spectral Baseline Model (IMS, 6 PCA)
------------------------------------------------------------
Pure Spectral Baseline (IMS, 6 PCA) Metrics:
[Train] R²: 0.965497 | MAE: 0.022065 | MSE: 0.000882 | MRE: 0.024289 | RMSE: 0.029703
[Val] R²: 0.899912 | MAE: 0.034302 | MSE: 0.002139 | MRE: 0.038647 | RMSE: 0.046251
[Test] R²: 0.816638 | MAE: 0.044819 | MSE: 0.003045 | MRE: 0.048980 | RMSE: 0.055178
Train-Val Gap: 0.065584 | Train-Test Gap: 0.148859 | Valid: True

Step 3/4: Fused Model with Best Parameters (K=2)
------------------------------------------------------------
Selected 

In [7]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
import pandas as pd
import random

np.random.seed(42)
random.seed(42)

def build_reg_model(params):
    return RandomForestRegressor(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        min_samples_split=params['min_samples_split'],
        min_samples_leaf=params['min_samples_leaf'],
        max_features=params['max_features'],
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )

def mean_relative_error(y_true, y_pred):
    mask = y_true != 0
    if np.sum(mask) == 0:
        return 0.0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mre = mean_relative_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'mae': mae, 'mse': mse, 'rmse': rmse, 'mre': mre, 'r2': r2}

print("=" * 60)
print("Step 1/4: Data Preprocessing (IMS Prediction)")
print("-" * 60)

pattern = pd.read_csv('4-pattern2.csv', header=None, encoding='utf-8-sig').values.astype('float64')
pattern = np.where(np.isinf(pattern), np.nan, pattern)
mean_val = np.nanmean(pattern) if not np.isnan(np.nanmean(pattern)) else 0
pattern = np.nan_to_num(pattern, nan=mean_val)

min_vals = np.min(pattern, axis=0)
max_vals = np.max(pattern, axis=0)
range_vals = np.where(max_vals - min_vals == 0, 1, max_vals - min_vals)
pattern_normalized = (pattern - min_vals) / range_vals

scaler_spectral = StandardScaler()
pattern_scaled = scaler_spectral.fit_transform(pattern_normalized)

print("\nLoading image features...")
image_features = pd.read_csv('image_features_mobilenet_large.csv', header=None).values.astype('float64')
scaler_image = StandardScaler()
image_features_scaled = scaler_image.fit_transform(image_features)

label_data = pd.read_csv('label_rIMS2-2.csv', header=None, encoding='utf-8-sig').values.astype('float64')
groups = label_data[:, 0]
label_c_original = label_data[:, 1]
label_c = np.exp(label_c_original)
total_samples = len(label_c)

unique_groups = np.unique(groups)
test_size_groups = int(0.2 * len(unique_groups))
train_groups, test_groups = train_test_split(
    unique_groups,
    test_size=test_size_groups,
    random_state=42
)

train_mask = np.isin(groups, train_groups)
test_mask = np.isin(groups, test_groups)
train_indices = np.where(train_mask)[0]
test_indices = np.where(test_mask)[0]

X_train_raw_spectral = pattern_scaled[train_mask]
X_test_raw_spectral = pattern_scaled[test_mask]
y_train = label_c[train_mask]
y_test = label_c[test_mask]

pca_spectral = PCA(n_components=6)
X_train_spectral_pca = pca_spectral.fit_transform(X_train_raw_spectral)
X_test_spectral_pca = pca_spectral.transform(X_test_raw_spectral)
print(f"Spectral PCA completed: 6 components retained (fixed)")

pca_image = PCA(n_components=0.95)
pattern_image_pca_full = pca_image.fit_transform(image_features_scaled)
image_pca_variance = pca_image.explained_variance_ratio_
print(f"Image PCA completed: {pca_image.n_components_} components retained, cumulative variance ratio: {np.sum(image_pca_variance):.4f}")
print(f"Top 5 image PC variance ratios: {[round(v, 4) for v in image_pca_variance[:5]]}")

X_train_image_pca_full = pattern_image_pca_full[train_mask]
X_test_image_pca_full = pattern_image_pca_full[test_mask]

GAP_THRESHOLD = 0.15

print("\n" + "=" * 60)
print("Step 2/4: Pure Spectral Baseline Model (IMS, 6 PCA)")
print("-" * 60)

ALL_PREDICTIONS = []

baseline_best_params = {
    'max_depth': None,
    'max_features': 'sqrt',
    'min_samples_leaf': 1,
    'min_samples_split': 2,
    'n_estimators': 200
}

X_tr_base, X_val_base, y_tr_base, y_val_base, tr_idx_base, val_idx_base = train_test_split(
    X_train_spectral_pca, y_train, train_indices,
    test_size=0.2,
    random_state=42
)
baseline_model = build_reg_model(baseline_best_params)
baseline_model.fit(X_tr_base, y_tr_base)

y_tr_pred_base = baseline_model.predict(X_tr_base)
y_val_pred_base = baseline_model.predict(X_val_base)
y_train_pred_base = baseline_model.predict(X_train_spectral_pca)
y_test_pred_base = baseline_model.predict(X_test_spectral_pca)

tr_metrics_base = calculate_metrics(y_tr_base, y_tr_pred_base)
val_metrics_base = calculate_metrics(y_val_base, y_val_pred_base)
train_metrics_base = calculate_metrics(y_train, y_train_pred_base)
test_metrics_base = calculate_metrics(y_test, y_test_pred_base)

train_val_gap_base = abs(train_metrics_base['r2'] - val_metrics_base['r2'])
train_test_gap_base = abs(train_metrics_base['r2'] - test_metrics_base['r2'])
is_valid_base = (train_val_gap_base < GAP_THRESHOLD) and (train_test_gap_base < GAP_THRESHOLD)

BASELINE_RESULT = {
    'Train_R2': round(train_metrics_base['r2'], 6),
    'Train_MAE': round(train_metrics_base['mae'], 6),
    'Train_MSE': round(train_metrics_base['mse'], 6),
    'Train_MRE': round(train_metrics_base['mre'], 6),
    'Train_RMSE': round(train_metrics_base['rmse'], 6),
    'Val_R2': round(val_metrics_base['r2'], 6),
    'Val_MAE': round(val_metrics_base['mae'], 6),
    'Val_MSE': round(val_metrics_base['mse'], 6),
    'Val_MRE': round(val_metrics_base['mre'], 6),
    'Val_RMSE': round(val_metrics_base['rmse'], 6),
    'Test_R2': round(test_metrics_base['r2'], 6),
    'Test_MAE': round(test_metrics_base['mae'], 6),
    'Test_MSE': round(test_metrics_base['mse'], 6),
    'Test_MRE': round(test_metrics_base['mre'], 6),
    'Test_RMSE': round(test_metrics_base['rmse'], 6),
    'Train_Val_Gap': round(train_val_gap_base, 6),
    'Train_Test_Gap': round(train_test_gap_base, 6),
    'Best_Params': str(baseline_best_params),
    'Is_Valid': is_valid_base
}

print("Pure Spectral Baseline (IMS, 6 PCA) Metrics:")
print(f"[Train] R²: {BASELINE_RESULT['Train_R2']:.6f} | MAE: {BASELINE_RESULT['Train_MAE']:.6f} | MSE: {BASELINE_RESULT['Train_MSE']:.6f} | MRE: {BASELINE_RESULT['Train_MRE']:.6f} | RMSE: {BASELINE_RESULT['Train_RMSE']:.6f}")
print(f"[Val] R²: {BASELINE_RESULT['Val_R2']:.6f} | MAE: {BASELINE_RESULT['Val_MAE']:.6f} | MSE: {BASELINE_RESULT['Val_MSE']:.6f} | MRE: {BASELINE_RESULT['Val_MRE']:.6f} | RMSE: {BASELINE_RESULT['Val_RMSE']:.6f}")
print(f"[Test] R²: {BASELINE_RESULT['Test_R2']:.6f} | MAE: {BASELINE_RESULT['Test_MAE']:.6f} | MSE: {BASELINE_RESULT['Test_MSE']:.6f} | MRE: {BASELINE_RESULT['Test_MRE']:.6f} | RMSE: {BASELINE_RESULT['Test_RMSE']:.6f}")
print(f"Train-Val Gap: {BASELINE_RESULT['Train_Val_Gap']:.6f} | Train-Test Gap: {BASELINE_RESULT['Train_Test_Gap']:.6f} | Valid: {BASELINE_RESULT['Is_Valid']}")

all_pred_base = np.full(total_samples, np.nan)
data_type_base = np.full(total_samples, "Unused")
all_pred_base[tr_idx_base] = y_tr_pred_base
all_pred_base[val_idx_base] = y_val_pred_base
all_pred_base[test_indices] = y_test_pred_base
data_type_base[tr_idx_base] = "Training Set"
data_type_base[val_idx_base] = "Validation Set"
data_type_base[test_indices] = "Test Set"
for idx in range(total_samples):
    ALL_PREDICTIONS.append({
        "Original_Index": idx + 1,
        "True_Label": round(label_c_original[idx], 6),
        "Model_Type": "Spectral_Baseline",
        "Predicted_Label": round(np.log(all_pred_base[idx]), 6) if not np.isnan(all_pred_base[idx]) else np.nan,
        "Data_Set_Type": data_type_base[idx]
    })

print("\n" + "=" * 60)
print("Step 3/4: Fused Model with Best Parameters (K=2)")
print("-" * 60)

best_k = 2
best_selected_indices = [6, 8]
best_params = {'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 300}

selected_variance = image_pca_variance[best_selected_indices]
cumulative_variance = np.sum(selected_variance)
print(f"Selected image PC indices: {best_selected_indices}")
print(f"Selected PC variance ratios: {[round(v, 4) for v in selected_variance]}")
print(f"Cumulative variance ratio: {cumulative_variance:.4f}")

selector = SelectKBest(score_func=f_regression, k=best_k)
X_train_image_selected = selector.fit_transform(X_train_image_pca_full, y_train)
X_test_image_selected = selector.transform(X_test_image_pca_full)

X_train_fused = np.hstack([X_train_spectral_pca, X_train_image_selected])
X_test_fused = np.hstack([X_test_spectral_pca, X_test_image_selected])

X_tr, X_val, y_tr, y_val, tr_idx, val_idx = train_test_split(
    X_train_fused, y_train, train_indices,
    test_size=0.2,
    random_state=42
)

final_model = build_reg_model(best_params)
final_model.fit(X_tr, y_tr)

y_tr_pred = final_model.predict(X_tr)
y_val_pred = final_model.predict(X_val)
y_train_pred = final_model.predict(X_train_fused)
y_test_pred = final_model.predict(X_test_fused)

tr_metrics = calculate_metrics(y_tr, y_tr_pred)
val_metrics = calculate_metrics(y_val, y_val_pred)
train_metrics = calculate_metrics(y_train, y_train_pred)
test_metrics = calculate_metrics(y_test, y_test_pred)

train_val_gap = abs(train_metrics['r2'] - val_metrics['r2'])
train_test_gap = abs(train_metrics['r2'] - test_metrics['r2'])
is_valid = (train_val_gap < GAP_THRESHOLD) and (train_test_gap < GAP_THRESHOLD)

BEST_OVERALL = {
    'K': best_k,
    'Selected_Image_PCs': str(best_selected_indices),
    'Cumulative_Variance_Ratio': round(cumulative_variance, 4),
    'Best_Params': str(best_params),
    'CV_R2': 'N/A (direct best params)',
    'Train_R2': round(train_metrics['r2'], 6),
    'Train_MAE': round(train_metrics['mae'], 6),
    'Train_MSE': round(train_metrics['mse'], 6),
    'Train_MRE': round(train_metrics['mre'], 6),
    'Train_RMSE': round(train_metrics['rmse'], 6),
    'Val_R2': round(val_metrics['r2'], 6),
    'Val_MAE': round(val_metrics['mae'], 6),
    'Val_MSE': round(val_metrics['mse'], 6),
    'Val_MRE': round(val_metrics['mre'], 6),
    'Val_RMSE': round(val_metrics['rmse'], 6),
    'Test_R2': round(test_metrics['r2'], 6),
    'Test_MAE': round(test_metrics['mae'], 6),
    'Test_MSE': round(test_metrics['mse'], 6),
    'Test_MRE': round(test_metrics['mre'], 6),
    'Test_RMSE': round(test_metrics['rmse'], 6),
    'Train_Val_Gap': round(train_val_gap, 6),
    'Train_Test_Gap': round(train_test_gap, 6),
    'Is_Valid': is_valid
}

all_pred = np.full(total_samples, np.nan)
data_type = np.full(total_samples, "Unused")
all_pred[tr_idx] = y_tr_pred
all_pred[val_idx] = y_val_pred
all_pred[test_indices] = y_test_pred
data_type[tr_idx] = "Training Set"
data_type[val_idx] = "Validation Set"
data_type[test_indices] = "Test Set"
for idx in range(total_samples):
    ALL_PREDICTIONS.append({
        "Original_Index": idx + 1,
        "True_Label": round(label_c_original[idx], 6),
        "Model_Type": "Fused_K=2",
        "Predicted_Label": round(np.log(all_pred[idx]), 6) if not np.isnan(all_pred[idx]) else np.nan,
        "Data_Set_Type": data_type[idx]
    })

print(f"Test R²: {test_metrics['r2']:.6f} | Train-Test Gap: {train_test_gap:.6f} | Valid: {is_valid}")

print("\n" + "=" * 60)
print("Step 4/4: Final Results Summary (IMS)")
print("-" * 60)

results_list = [BEST_OVERALL]
results_df = pd.DataFrame(results_list)
predictions_df = pd.DataFrame(ALL_PREDICTIONS)
predictions_df = predictions_df.sort_values(by=['Original_Index', 'Model_Type']).reset_index(drop=True)

summary_cols = ['K', 'Cumulative_Variance_Ratio', 'CV_R2', 'Test_R2', 'Test_MAE', 'Test_MSE', 'Test_RMSE', 'Train_Test_Gap', 'Is_Valid']
print(results_df[summary_cols].to_string(index=False))

print("\n" + "=" * 60)
print("Best Overall Fused Model Results (IMS)")
print("-" * 60)
print(f"Optimal K value: {BEST_OVERALL['K']}")
print(f"Selected image PC indices: {BEST_OVERALL['Selected_Image_PCs']}")
print(f"Cumulative variance ratio: {BEST_OVERALL['Cumulative_Variance_Ratio']:.4f}")
print(f"Optimal parameters: {BEST_OVERALL['Best_Params']}")
print("\nDetailed Performance Metrics:")
print(f"[Train] R²: {BEST_OVERALL['Train_R2']:.6f} | MAE: {BEST_OVERALL['Train_MAE']:.6f} | MSE: {BEST_OVERALL['Train_MSE']:.6f} | MRE: {BEST_OVERALL['Train_MRE']:.6f} | RMSE: {BEST_OVERALL['Train_RMSE']:.6f}")
print(f"[Val] R²: {BEST_OVERALL['Val_R2']:.6f} | MAE: {BEST_OVERALL['Val_MAE']:.6f} | MSE: {BEST_OVERALL['Val_MSE']:.6f} | MRE: {BEST_OVERALL['Val_MRE']:.6f} | RMSE: {BEST_OVERALL['Val_RMSE']:.6f}")
print(f"[Test] R²: {BEST_OVERALL['Test_R2']:.6f} | MAE: {BEST_OVERALL['Test_MAE']:.6f} | MSE: {BEST_OVERALL['Test_MSE']:.6f} | MRE: {BEST_OVERALL['Test_MRE']:.6f} | RMSE: {BEST_OVERALL['Test_RMSE']:.6f}")
print(f"Train-Val Gap: {BEST_OVERALL['Train_Val_Gap']:.6f} | Train-Test Gap: {BEST_OVERALL['Train_Test_Gap']:.6f} | Valid: {BEST_OVERALL['Is_Valid']}")

train_improve_r2 = (BEST_OVERALL['Train_R2'] - BASELINE_RESULT['Train_R2']) * 100
train_improve_mae = (BASELINE_RESULT['Train_MAE'] - BEST_OVERALL['Train_MAE']) * 100
train_improve_mse = (BASELINE_RESULT['Train_MSE'] - BEST_OVERALL['Train_MSE']) * 100
train_improve_mre = (BASELINE_RESULT['Train_MRE'] - BEST_OVERALL['Train_MRE']) * 100
train_improve_rmse = (BASELINE_RESULT['Train_RMSE'] - BEST_OVERALL['Train_RMSE']) * 100

val_improve_r2 = (BEST_OVERALL['Val_R2'] - BASELINE_RESULT['Val_R2']) * 100
val_improve_mae = (BASELINE_RESULT['Val_MAE'] - BEST_OVERALL['Val_MAE']) * 100
val_improve_mse = (BASELINE_RESULT['Val_MSE'] - BEST_OVERALL['Val_MSE']) * 100
val_improve_mre = (BASELINE_RESULT['Val_MRE'] - BEST_OVERALL['Val_MRE']) * 100
val_improve_rmse = (BASELINE_RESULT['Val_RMSE'] - BEST_OVERALL['Val_RMSE']) * 100

test_improve_r2 = (BEST_OVERALL['Test_R2'] - BASELINE_RESULT['Test_R2']) * 100
test_improve_mae = (BASELINE_RESULT['Test_MAE'] - BEST_OVERALL['Test_MAE']) * 100
test_improve_mse = (BASELINE_RESULT['Test_MSE'] - BEST_OVERALL['Test_MSE']) * 100
test_improve_mre = (BASELINE_RESULT['Test_MRE'] - BEST_OVERALL['Test_MRE']) * 100
test_improve_rmse = (BASELINE_RESULT['Test_RMSE'] - BEST_OVERALL['Test_RMSE']) * 100

print("\nComparison with Pure Spectral Baseline (All Datasets):")
print("-"*50)
print(f"Training Set")
print(f"Baseline Train R²: {BASELINE_RESULT['Train_R2']:.6f} | Best Fused Train R²: {BEST_OVERALL['Train_R2']:.6f}")
print(f"Improvement: R² {train_improve_r2:+.2f}% | MAE {train_improve_mae:+.2f}% | MSE {train_improve_mse:+.2f}% | MRE {train_improve_mre:+.2f}% | RMSE {train_improve_rmse:+.2f}%")

print(f"\nValidation Set")
print(f"Baseline Val R²: {BASELINE_RESULT['Val_R2']:.6f} | Best Fused Val R²: {BEST_OVERALL['Val_R2']:.6f}")
print(f"Improvement: R² {val_improve_r2:+.2f}% | MAE {val_improve_mae:+.2f}% | MSE {val_improve_mse:+.2f}% | MRE {val_improve_mre:+.2f}% | RMSE {val_improve_rmse:+.2f}%")

print(f"\nTest Set")
print(f"Baseline Test R²: {BASELINE_RESULT['Test_R2']:.6f} | Best Fused Test R²: {BEST_OVERALL['Test_R2']:.6f}")
print(f"Improvement: R² {test_improve_r2:+.2f}% | MAE {test_improve_mae:+.2f}% | MSE {test_improve_mse:+.2f}% | MRE {test_improve_mre:+.2f}% | RMSE {test_improve_rmse:+.2f}%")

print("\n" + "=" * 60)
print("Step 5/5: Robustness Verification - Input Feature Noise Injection (10%)")
print("-" * 60)

noise_level = 0.10
spectral_test_std = np.std(X_test_spectral_pca, axis=0)
fused_test_std = np.std(X_test_fused, axis=0)

noise_spectral = np.random.normal(0, noise_level * spectral_test_std, X_test_spectral_pca.shape)
X_test_spectral_noisy = X_test_spectral_pca + noise_spectral
noise_fused = np.random.normal(0, noise_level * fused_test_std, X_test_fused.shape)
X_test_fused_noisy = X_test_fused + noise_fused

y_pred_base_noisy = baseline_model.predict(X_test_spectral_noisy)
metrics_base_noisy = calculate_metrics(y_test, y_pred_base_noisy)
metrics_base_noisy['Noise_Level'] = noise_level

y_pred_fused_noisy = final_model.predict(X_test_fused_noisy)
metrics_fused_noisy = calculate_metrics(y_test, y_pred_fused_noisy)
metrics_fused_noisy['Noise_Level'] = noise_level

print("\nNoise Injection Results (10%):")
print(f"Baseline - R²: {metrics_base_noisy['r2']:.6f}, MAE: {metrics_base_noisy['mae']:.6f}, RMSE: {metrics_base_noisy['rmse']:.6f}, MRE: {metrics_base_noisy['mre']:.6f}")
print(f"Fused   - R²: {metrics_fused_noisy['r2']:.6f}, MAE: {metrics_fused_noisy['mae']:.6f}, RMSE: {metrics_fused_noisy['rmse']:.6f}, MRE: {metrics_fused_noisy['mre']:.6f}")

noise_pred_baseline = pd.DataFrame({
    'Original_Index': test_indices + 1,
    'True_Exp': y_test,
    'Predicted_Exp_Original': y_test_pred_base,
    'Predicted_Exp_Noisy': y_pred_base_noisy
})
noise_pred_baseline['Model'] = 'Baseline'

noise_pred_fused = pd.DataFrame({
    'Original_Index': test_indices + 1,
    'True_Exp': y_test,
    'Predicted_Exp_Original': y_test_pred,
    'Predicted_Exp_Noisy': y_pred_fused_noisy
})
noise_pred_fused['Model'] = 'Fused'

noise_pred_df = pd.concat([noise_pred_baseline, noise_pred_fused], ignore_index=True)
noise_pred_df = noise_pred_df[['Model', 'Original_Index', 'True_Exp', 'Predicted_Exp_Original', 'Predicted_Exp_Noisy']]

df_noise_baseline = pd.DataFrame([metrics_base_noisy])
df_noise_fused = pd.DataFrame([metrics_fused_noisy])

with pd.ExcelWriter('Image_Feature_K_Selection_Results_IMS_Tuned_Noise10.xlsx', engine='openpyxl') as writer:
    pd.DataFrame([BASELINE_RESULT]).to_excel(writer, sheet_name='Spectral_Baseline', index=False)
    results_df.to_excel(writer, sheet_name='Best_Fused_Result', index=False)
    predictions_df.to_excel(writer, sheet_name='All_Sample_Predictions', index=False)
    df_noise_baseline.to_excel(writer, sheet_name='Noise_Test_Baseline', index=False)
    df_noise_fused.to_excel(writer, sheet_name='Noise_Test_Fused', index=False)
    noise_pred_df.to_excel(writer, sheet_name='Noise_Predictions', index=False)

print(f"\nAll results saved to: Image_Feature_K_Selection_Results_IMS_Tuned_Noise10.xlsx")
print("=" * 60)

Step 1/4: Data Preprocessing (IMS Prediction)
------------------------------------------------------------

Loading image features...
Spectral PCA completed: 6 components retained (fixed)
Image PCA completed: 186 components retained, cumulative variance ratio: 0.9503
Top 5 image PC variance ratios: [np.float64(0.1658), np.float64(0.0762), np.float64(0.0614), np.float64(0.0468), np.float64(0.0415)]

Step 2/4: Pure Spectral Baseline Model (IMS, 6 PCA)
------------------------------------------------------------
Pure Spectral Baseline (IMS, 6 PCA) Metrics:
[Train] R²: 0.965497 | MAE: 0.022065 | MSE: 0.000882 | MRE: 0.024289 | RMSE: 0.029703
[Val] R²: 0.899912 | MAE: 0.034302 | MSE: 0.002139 | MRE: 0.038647 | RMSE: 0.046251
[Test] R²: 0.816638 | MAE: 0.044819 | MSE: 0.003045 | MRE: 0.048980 | RMSE: 0.055178
Train-Val Gap: 0.065584 | Train-Test Gap: 0.148859 | Valid: True

Step 3/4: Fused Model with Best Parameters (K=2)
------------------------------------------------------------
Selected 

In [5]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
import pandas as pd
import random

np.random.seed(42)
random.seed(42)

def build_reg_model(params):
    return RandomForestRegressor(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        min_samples_split=params['min_samples_split'],
        min_samples_leaf=params['min_samples_leaf'],
        max_features=params['max_features'],
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )

def mean_relative_error(y_true, y_pred):
    mask = y_true != 0
    if np.sum(mask) == 0:
        return 0.0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mre = mean_relative_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'mae': mae, 'mse': mse, 'rmse': rmse, 'mre': mre, 'r2': r2}

print("=" * 60)
print("Step 1/6: Data Preprocessing (IMS Prediction)")
print("-" * 60)

pattern = pd.read_csv('4-pattern2.csv', header=None, encoding='utf-8-sig').values.astype('float64')
pattern = np.where(np.isinf(pattern), np.nan, pattern)
mean_val = np.nanmean(pattern) if not np.isnan(np.nanmean(pattern)) else 0
pattern = np.nan_to_num(pattern, nan=mean_val)

min_vals = np.min(pattern, axis=0)
max_vals = np.max(pattern, axis=0)
range_vals = np.where(max_vals - min_vals == 0, 1, max_vals - min_vals)
pattern_normalized = (pattern - min_vals) / range_vals

scaler_spectral = StandardScaler()
pattern_scaled = scaler_spectral.fit_transform(pattern_normalized)

print("\nLoading image features...")
image_features = pd.read_csv('image_features_mobilenet_large.csv', header=None).values.astype('float64')
scaler_image = StandardScaler()
image_features_scaled = scaler_image.fit_transform(image_features)

label_data = pd.read_csv('label_rIMS2-2.csv', header=None, encoding='utf-8-sig').values.astype('float64')
groups = label_data[:, 0]
label_c_original = label_data[:, 1]
label_c = np.exp(label_c_original)
total_samples = len(label_c)

unique_groups = np.unique(groups)
test_size_groups = int(0.2 * len(unique_groups))
train_groups, test_groups = train_test_split(
    unique_groups,
    test_size=test_size_groups,
    random_state=42
)

train_mask = np.isin(groups, train_groups)
test_mask = np.isin(groups, test_groups)
train_indices = np.where(train_mask)[0]
test_indices = np.where(test_mask)[0]

X_train_raw_spectral = pattern_scaled[train_mask]
X_test_raw_spectral = pattern_scaled[test_mask]
y_train = label_c[train_mask]
y_test = label_c[test_mask]

pca_spectral = PCA(n_components=6)
X_train_spectral_pca = pca_spectral.fit_transform(X_train_raw_spectral)
X_test_spectral_pca = pca_spectral.transform(X_test_raw_spectral)
print(f"Spectral PCA completed: 6 components retained (fixed)")

pca_image = PCA(n_components=0.95)
pattern_image_pca_full = pca_image.fit_transform(image_features_scaled)
image_pca_variance = pca_image.explained_variance_ratio_
print(f"Image PCA completed: {pca_image.n_components_} components retained, cumulative variance ratio: {np.sum(image_pca_variance):.4f}")
print(f"Top 5 image PC variance ratios: {[round(v, 4) for v in image_pca_variance[:5]]}")

X_train_image_pca_full = pattern_image_pca_full[train_mask]
X_test_image_pca_full = pattern_image_pca_full[test_mask]

GAP_THRESHOLD = 0.15

print("\n" + "=" * 60)
print("Step 2/6: Pure Spectral Baseline Model (IMS, 6 PCA)")
print("-" * 60)

ALL_PREDICTIONS = []

baseline_best_params = {
    'max_depth': None,
    'max_features': 'sqrt',
    'min_samples_leaf': 1,
    'min_samples_split': 2,
    'n_estimators': 200
}

X_tr_base, X_val_base, y_tr_base, y_val_base, tr_idx_base, val_idx_base = train_test_split(
    X_train_spectral_pca, y_train, train_indices,
    test_size=0.2,
    random_state=42
)
baseline_model = build_reg_model(baseline_best_params)
baseline_model.fit(X_tr_base, y_tr_base)

y_tr_pred_base = baseline_model.predict(X_tr_base)
y_val_pred_base = baseline_model.predict(X_val_base)
y_train_pred_base = baseline_model.predict(X_train_spectral_pca)
y_test_pred_base = baseline_model.predict(X_test_spectral_pca)

tr_metrics_base = calculate_metrics(y_tr_base, y_tr_pred_base)
val_metrics_base = calculate_metrics(y_val_base, y_val_pred_base)
train_metrics_base = calculate_metrics(y_train, y_train_pred_base)
test_metrics_base = calculate_metrics(y_test, y_test_pred_base)

train_val_gap_base = abs(train_metrics_base['r2'] - val_metrics_base['r2'])
train_test_gap_base = abs(train_metrics_base['r2'] - test_metrics_base['r2'])
is_valid_base = (train_val_gap_base < GAP_THRESHOLD) and (train_test_gap_base < GAP_THRESHOLD)

BASELINE_RESULT = {
    'Train_R2': round(train_metrics_base['r2'], 6),
    'Train_MAE': round(train_metrics_base['mae'], 6),
    'Train_MSE': round(train_metrics_base['mse'], 6),
    'Train_MRE': round(train_metrics_base['mre'], 6),
    'Train_RMSE': round(train_metrics_base['rmse'], 6),
    'Val_R2': round(val_metrics_base['r2'], 6),
    'Val_MAE': round(val_metrics_base['mae'], 6),
    'Val_MSE': round(val_metrics_base['mse'], 6),
    'Val_MRE': round(val_metrics_base['mre'], 6),
    'Val_RMSE': round(val_metrics_base['rmse'], 6),
    'Test_R2': round(test_metrics_base['r2'], 6),
    'Test_MAE': round(test_metrics_base['mae'], 6),
    'Test_MSE': round(test_metrics_base['mse'], 6),
    'Test_MRE': round(test_metrics_base['mre'], 6),
    'Test_RMSE': round(test_metrics_base['rmse'], 6),
    'Train_Val_Gap': round(train_val_gap_base, 6),
    'Train_Test_Gap': round(train_test_gap_base, 6),
    'Best_Params': str(baseline_best_params),
    'Is_Valid': is_valid_base
}

print("Pure Spectral Baseline (IMS, 6 PCA) Metrics:")
print(f"[Train] R²: {BASELINE_RESULT['Train_R2']:.6f} | MAE: {BASELINE_RESULT['Train_MAE']:.6f} | MSE: {BASELINE_RESULT['Train_MSE']:.6f} | MRE: {BASELINE_RESULT['Train_MRE']:.6f} | RMSE: {BASELINE_RESULT['Train_RMSE']:.6f}")
print(f"[Val] R²: {BASELINE_RESULT['Val_R2']:.6f} | MAE: {BASELINE_RESULT['Val_MAE']:.6f} | MSE: {BASELINE_RESULT['Val_MSE']:.6f} | MRE: {BASELINE_RESULT['Val_MRE']:.6f} | RMSE: {BASELINE_RESULT['Val_RMSE']:.6f}")
print(f"[Test] R²: {BASELINE_RESULT['Test_R2']:.6f} | MAE: {BASELINE_RESULT['Test_MAE']:.6f} | MSE: {BASELINE_RESULT['Test_MSE']:.6f} | MRE: {BASELINE_RESULT['Test_MRE']:.6f} | RMSE: {BASELINE_RESULT['Test_RMSE']:.6f}")
print(f"Train-Val Gap: {BASELINE_RESULT['Train_Val_Gap']:.6f} | Train-Test Gap: {BASELINE_RESULT['Train_Test_Gap']:.6f} | Valid: {BASELINE_RESULT['Is_Valid']}")

all_pred_base = np.full(total_samples, np.nan)
data_type_base = np.full(total_samples, "Unused")
all_pred_base[tr_idx_base] = y_tr_pred_base
all_pred_base[val_idx_base] = y_val_pred_base
all_pred_base[test_indices] = y_test_pred_base
data_type_base[tr_idx_base] = "Training Set"
data_type_base[val_idx_base] = "Validation Set"
data_type_base[test_indices] = "Test Set"
for idx in range(total_samples):
    ALL_PREDICTIONS.append({
        "Original_Index": idx + 1,
        "True_Label": round(label_c_original[idx], 6),
        "Model_Type": "Spectral_Baseline",
        "Predicted_Label": round(np.log(all_pred_base[idx]), 6) if not np.isnan(all_pred_base[idx]) else np.nan,
        "Data_Set_Type": data_type_base[idx]
    })

print("\n" + "=" * 60)
print("Step 3/6: Fused Model with Best Parameters (K=2)")
print("-" * 60)

best_k = 2
best_selected_indices = [6, 8]
best_params = {'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 300}

selected_variance = image_pca_variance[best_selected_indices]
cumulative_variance = np.sum(selected_variance)
print(f"Selected image PC indices: {best_selected_indices}")
print(f"Selected PC variance ratios: {[round(v, 4) for v in selected_variance]}")
print(f"Cumulative variance ratio: {cumulative_variance:.4f}")

selector = SelectKBest(score_func=f_regression, k=best_k)
X_train_image_selected = selector.fit_transform(X_train_image_pca_full, y_train)
X_test_image_selected = selector.transform(X_test_image_pca_full)

X_train_fused = np.hstack([X_train_spectral_pca, X_train_image_selected])
X_test_fused = np.hstack([X_test_spectral_pca, X_test_image_selected])

X_tr, X_val, y_tr, y_val, tr_idx, val_idx = train_test_split(
    X_train_fused, y_train, train_indices,
    test_size=0.2,
    random_state=42
)

final_model = build_reg_model(best_params)
final_model.fit(X_tr, y_tr)

y_tr_pred = final_model.predict(X_tr)
y_val_pred = final_model.predict(X_val)
y_train_pred = final_model.predict(X_train_fused)
y_test_pred = final_model.predict(X_test_fused)

tr_metrics = calculate_metrics(y_tr, y_tr_pred)
val_metrics = calculate_metrics(y_val, y_val_pred)
train_metrics = calculate_metrics(y_train, y_train_pred)
test_metrics = calculate_metrics(y_test, y_test_pred)

train_val_gap = abs(train_metrics['r2'] - val_metrics['r2'])
train_test_gap = abs(train_metrics['r2'] - test_metrics['r2'])
is_valid = (train_val_gap < GAP_THRESHOLD) and (train_test_gap < GAP_THRESHOLD)

BEST_OVERALL = {
    'K': best_k,
    'Selected_Image_PCs': str(best_selected_indices),
    'Cumulative_Variance_Ratio': round(cumulative_variance, 4),
    'Best_Params': str(best_params),
    'CV_R2': 'N/A (direct best params)',
    'Train_R2': round(train_metrics['r2'], 6),
    'Train_MAE': round(train_metrics['mae'], 6),
    'Train_MSE': round(train_metrics['mse'], 6),
    'Train_MRE': round(train_metrics['mre'], 6),
    'Train_RMSE': round(train_metrics['rmse'], 6),
    'Val_R2': round(val_metrics['r2'], 6),
    'Val_MAE': round(val_metrics['mae'], 6),
    'Val_MSE': round(val_metrics['mse'], 6),
    'Val_MRE': round(val_metrics['mre'], 6),
    'Val_RMSE': round(val_metrics['rmse'], 6),
    'Test_R2': round(test_metrics['r2'], 6),
    'Test_MAE': round(test_metrics['mae'], 6),
    'Test_MSE': round(test_metrics['mse'], 6),
    'Test_MRE': round(test_metrics['mre'], 6),
    'Test_RMSE': round(test_metrics['rmse'], 6),
    'Train_Val_Gap': round(train_val_gap, 6),
    'Train_Test_Gap': round(train_test_gap, 6),
    'Is_Valid': is_valid
}

all_pred = np.full(total_samples, np.nan)
data_type = np.full(total_samples, "Unused")
all_pred[tr_idx] = y_tr_pred
all_pred[val_idx] = y_val_pred
all_pred[test_indices] = y_test_pred
data_type[tr_idx] = "Training Set"
data_type[val_idx] = "Validation Set"
data_type[test_indices] = "Test Set"
for idx in range(total_samples):
    ALL_PREDICTIONS.append({
        "Original_Index": idx + 1,
        "True_Label": round(label_c_original[idx], 6),
        "Model_Type": "Fused_K=2",
        "Predicted_Label": round(np.log(all_pred[idx]), 6) if not np.isnan(all_pred[idx]) else np.nan,
        "Data_Set_Type": data_type[idx]
    })

print(f"Test R²: {test_metrics['r2']:.6f} | Train-Test Gap: {train_test_gap:.6f} | Valid: {is_valid}")

print("\n" + "=" * 60)
print("Step 4/6: Final Results Summary (IMS)")
print("-" * 60)

results_list = [BEST_OVERALL]
results_df = pd.DataFrame(results_list)
predictions_df = pd.DataFrame(ALL_PREDICTIONS)
predictions_df = predictions_df.sort_values(by=['Original_Index', 'Model_Type']).reset_index(drop=True)

summary_cols = ['K', 'Cumulative_Variance_Ratio', 'CV_R2', 'Test_R2', 'Test_MAE', 'Test_MSE', 'Test_RMSE', 'Train_Test_Gap', 'Is_Valid']
print(results_df[summary_cols].to_string(index=False))

print("\n" + "=" * 60)
print("Best Overall Fused Model Results (IMS)")
print("-" * 60)
print(f"Optimal K value: {BEST_OVERALL['K']}")
print(f"Selected image PC indices: {BEST_OVERALL['Selected_Image_PCs']}")
print(f"Cumulative variance ratio: {BEST_OVERALL['Cumulative_Variance_Ratio']:.4f}")
print(f"Optimal parameters: {BEST_OVERALL['Best_Params']}")
print("\nDetailed Performance Metrics:")
print(f"[Train] R²: {BEST_OVERALL['Train_R2']:.6f} | MAE: {BEST_OVERALL['Train_MAE']:.6f} | MSE: {BEST_OVERALL['Train_MSE']:.6f} | MRE: {BEST_OVERALL['Train_MRE']:.6f} | RMSE: {BEST_OVERALL['Train_RMSE']:.6f}")
print(f"[Val] R²: {BEST_OVERALL['Val_R2']:.6f} | MAE: {BEST_OVERALL['Val_MAE']:.6f} | MSE: {BEST_OVERALL['Val_MSE']:.6f} | MRE: {BEST_OVERALL['Val_MRE']:.6f} | RMSE: {BEST_OVERALL['Val_RMSE']:.6f}")
print(f"[Test] R²: {BEST_OVERALL['Test_R2']:.6f} | MAE: {BEST_OVERALL['Test_MAE']:.6f} | MSE: {BEST_OVERALL['Test_MSE']:.6f} | MRE: {BEST_OVERALL['Test_MRE']:.6f} | RMSE: {BEST_OVERALL['Test_RMSE']:.6f}")
print(f"Train-Val Gap: {BEST_OVERALL['Train_Val_Gap']:.6f} | Train-Test Gap: {BEST_OVERALL['Train_Test_Gap']:.6f} | Valid: {BEST_OVERALL['Is_Valid']}")

train_improve_r2 = (BEST_OVERALL['Train_R2'] - BASELINE_RESULT['Train_R2']) * 100
train_improve_mae = (BASELINE_RESULT['Train_MAE'] - BEST_OVERALL['Train_MAE']) * 100
train_improve_mse = (BASELINE_RESULT['Train_MSE'] - BEST_OVERALL['Train_MSE']) * 100
train_improve_mre = (BASELINE_RESULT['Train_MRE'] - BEST_OVERALL['Train_MRE']) * 100
train_improve_rmse = (BASELINE_RESULT['Train_RMSE'] - BEST_OVERALL['Train_RMSE']) * 100

val_improve_r2 = (BEST_OVERALL['Val_R2'] - BASELINE_RESULT['Val_R2']) * 100
val_improve_mae = (BASELINE_RESULT['Val_MAE'] - BEST_OVERALL['Val_MAE']) * 100
val_improve_mse = (BASELINE_RESULT['Val_MSE'] - BEST_OVERALL['Val_MSE']) * 100
val_improve_mre = (BASELINE_RESULT['Val_MRE'] - BEST_OVERALL['Val_MRE']) * 100
val_improve_rmse = (BASELINE_RESULT['Val_RMSE'] - BEST_OVERALL['Val_RMSE']) * 100

test_improve_r2 = (BEST_OVERALL['Test_R2'] - BASELINE_RESULT['Test_R2']) * 100
test_improve_mae = (BASELINE_RESULT['Test_MAE'] - BEST_OVERALL['Test_MAE']) * 100
test_improve_mse = (BASELINE_RESULT['Test_MSE'] - BEST_OVERALL['Test_MSE']) * 100
test_improve_mre = (BASELINE_RESULT['Test_MRE'] - BEST_OVERALL['Test_MRE']) * 100
test_improve_rmse = (BASELINE_RESULT['Test_RMSE'] - BEST_OVERALL['Test_RMSE']) * 100

print("\nComparison with Pure Spectral Baseline (All Datasets):")
print("-"*50)
print(f"Training Set")
print(f"Baseline Train R²: {BASELINE_RESULT['Train_R2']:.6f} | Best Fused Train R²: {BEST_OVERALL['Train_R2']:.6f}")
print(f"Improvement: R² {train_improve_r2:+.2f}% | MAE {train_improve_mae:+.2f}% | MSE {train_improve_mse:+.2f}% | MRE {train_improve_mre:+.2f}% | RMSE {train_improve_rmse:+.2f}%")

print(f"\nValidation Set")
print(f"Baseline Val R²: {BASELINE_RESULT['Val_R2']:.6f} | Best Fused Val R²: {BEST_OVERALL['Val_R2']:.6f}")
print(f"Improvement: R² {val_improve_r2:+.2f}% | MAE {val_improve_mae:+.2f}% | MSE {val_improve_mse:+.2f}% | MRE {val_improve_mre:+.2f}% | RMSE {val_improve_rmse:+.2f}%")

print(f"\nTest Set")
print(f"Baseline Test R²: {BASELINE_RESULT['Test_R2']:.6f} | Best Fused Test R²: {BEST_OVERALL['Test_R2']:.6f}")
print(f"Improvement: R² {test_improve_r2:+.2f}% | MAE {test_improve_mae:+.2f}% | MSE {test_improve_mse:+.2f}% | MRE {test_improve_mre:+.2f}% | RMSE {test_improve_rmse:+.2f}%")

print("\n" + "=" * 60)
print("Step 5/6: Robustness Verification - Input Feature Noise Injection")
print("-" * 60)

noise_levels = [0.0, 0.01, 0.05, 0.1, 0.2]
noise_results_baseline = []
noise_results_fused = []

spectral_test_std = np.std(X_test_spectral_pca, axis=0)
fused_test_std = np.std(X_test_fused, axis=0)

for level in noise_levels:
    if level == 0.0:
        X_test_spectral_noisy = X_test_spectral_pca.copy()
        X_test_fused_noisy = X_test_fused.copy()
    else:
        noise_spectral = np.random.normal(0, level * spectral_test_std, X_test_spectral_pca.shape)
        X_test_spectral_noisy = X_test_spectral_pca + noise_spectral
        noise_fused = np.random.normal(0, level * fused_test_std, X_test_fused.shape)
        X_test_fused_noisy = X_test_fused + noise_fused

    y_pred_base_noisy = baseline_model.predict(X_test_spectral_noisy)
    metrics_base = calculate_metrics(y_test, y_pred_base_noisy)
    metrics_base['Noise_Level'] = level
    noise_results_baseline.append(metrics_base)

    y_pred_fused_noisy = final_model.predict(X_test_fused_noisy)
    metrics_fused = calculate_metrics(y_test, y_pred_fused_noisy)
    metrics_fused['Noise_Level'] = level
    noise_results_fused.append(metrics_fused)

df_noise_baseline = pd.DataFrame(noise_results_baseline)
df_noise_fused = pd.DataFrame(noise_results_fused)

print("\nBaseline Model Noise Injection Results:")
print(df_noise_baseline[['Noise_Level', 'r2', 'mae', 'rmse', 'mre']].to_string(index=False))
print("\nFused Model Noise Injection Results:")
print(df_noise_fused[['Noise_Level', 'r2', 'mae', 'rmse', 'mre']].to_string(index=False))

print("\n" + "=" * 60)
print("Step 6/6: Adversarial Sample Test (Epsilon = 10%)")
print("-" * 60)

epsilon = 0.1
top_k = 3

baseline_importances = baseline_model.feature_importances_
baseline_top_indices = np.argsort(baseline_importances)[-top_k:]
print(f"Baseline top-{top_k} feature indices: {baseline_top_indices} (importances: {[round(baseline_importances[i], 4) for i in baseline_top_indices]})")

fused_importances = final_model.feature_importances_
fused_top_indices = np.argsort(fused_importances)[-top_k:]
print(f"Fused top-{top_k} feature indices: {fused_top_indices} (importances: {[round(fused_importances[i], 4) for i in fused_top_indices]})")

baseline_feat_std = np.std(X_test_spectral_pca, axis=0)
fused_feat_std = np.std(X_test_fused, axis=0)

X_test_baseline_adv = X_test_spectral_pca.copy()
X_test_fused_adv = X_test_fused.copy()

for idx in range(len(y_test)):
    for feat_idx in baseline_top_indices:
        delta = epsilon * baseline_feat_std[feat_idx]
        X_test_baseline_adv[idx, feat_idx] += delta * np.random.choice([-1, 1])
    for feat_idx in fused_top_indices:
        delta = epsilon * fused_feat_std[feat_idx]
        X_test_fused_adv[idx, feat_idx] += delta * np.random.choice([-1, 1])

y_pred_base_adv = baseline_model.predict(X_test_baseline_adv)
y_pred_fused_adv = final_model.predict(X_test_fused_adv)

metrics_base_adv = calculate_metrics(y_test, y_pred_base_adv)
metrics_fused_adv = calculate_metrics(y_test, y_pred_fused_adv)

print("\nAdversarial Test Metrics (Epsilon=10%):")
print(f"Baseline - R²: {metrics_base_adv['r2']:.6f}, MAE: {metrics_base_adv['mae']:.6f}, RMSE: {metrics_base_adv['rmse']:.6f}, MRE: {metrics_base_adv['mre']:.6f}")
print(f"Fused   - R²: {metrics_fused_adv['r2']:.6f}, MAE: {metrics_fused_adv['mae']:.6f}, RMSE: {metrics_fused_adv['rmse']:.6f}, MRE: {metrics_fused_adv['mre']:.6f}")

adv_baseline_df = pd.DataFrame({
    'Original_Index': test_indices + 1,
    'True_Label_Log': label_c_original[test_mask],
    'True_Exp': y_test,
    'Predicted_Log_Original': np.log(y_test_pred_base),
    'Predicted_Exp_Original': y_test_pred_base,
    'Predicted_Log_Adversarial': np.log(np.maximum(y_pred_base_adv, 1e-8)),
    'Predicted_Exp_Adversarial': y_pred_base_adv
})

adv_fused_df = pd.DataFrame({
    'Original_Index': test_indices + 1,
    'True_Label_Log': label_c_original[test_mask],
    'True_Exp': y_test,
    'Predicted_Log_Original': np.log(y_test_pred),
    'Predicted_Exp_Original': y_test_pred,
    'Predicted_Log_Adversarial': np.log(np.maximum(y_pred_fused_adv, 1e-8)),
    'Predicted_Exp_Adversarial': y_pred_fused_adv
})

adv_filename = f'Adversarial_Predictions_Epsilon{int(epsilon*100)}.xlsx'
with pd.ExcelWriter(adv_filename, engine='openpyxl') as writer:
    adv_baseline_df.to_excel(writer, sheet_name='Baseline_Adversarial', index=False)
    adv_fused_df.to_excel(writer, sheet_name='Fused_Adversarial', index=False)

print(f"\nAdversarial prediction details saved to: {adv_filename}")

with pd.ExcelWriter('Image_Feature_K_Selection_Results_IMS_Tuned.xlsx', engine='openpyxl') as writer:
    pd.DataFrame([BASELINE_RESULT]).to_excel(writer, sheet_name='Spectral_Baseline', index=False)
    results_df.to_excel(writer, sheet_name='Best_Fused_Result', index=False)
    predictions_df.to_excel(writer, sheet_name='All_Sample_Predictions', index=False)
    df_noise_baseline.to_excel(writer, sheet_name='Noise_Test_Baseline', index=False)
    df_noise_fused.to_excel(writer, sheet_name='Noise_Test_Fused', index=False)

print(f"All results saved to: Image_Feature_K_Selection_Results_IMS_Tuned.xlsx")
print("=" * 60)

Step 1/6: Data Preprocessing (IMS Prediction)
------------------------------------------------------------

Loading image features...
Spectral PCA completed: 6 components retained (fixed)
Image PCA completed: 186 components retained, cumulative variance ratio: 0.9503
Top 5 image PC variance ratios: [np.float64(0.1658), np.float64(0.0762), np.float64(0.0614), np.float64(0.0468), np.float64(0.0415)]

Step 2/6: Pure Spectral Baseline Model (IMS, 6 PCA)
------------------------------------------------------------
Pure Spectral Baseline (IMS, 6 PCA) Metrics:
[Train] R²: 0.965497 | MAE: 0.022065 | MSE: 0.000882 | MRE: 0.024289 | RMSE: 0.029703
[Val] R²: 0.899912 | MAE: 0.034302 | MSE: 0.002139 | MRE: 0.038647 | RMSE: 0.046251
[Test] R²: 0.816638 | MAE: 0.044819 | MSE: 0.003045 | MRE: 0.048980 | RMSE: 0.055178
Train-Val Gap: 0.065584 | Train-Test Gap: 0.148859 | Valid: True

Step 3/6: Fused Model with Best Parameters (K=2)
------------------------------------------------------------
Selected 